# ITC560 Assessment 4: Greenhouse IoT Analytics

This notebook analyses a simulated smart greenhouse dataset. The readings represent IoT telemetry from two greenhouse zones, including sensors, actuators, device health, and gateway/network measurements.

Run the variant selector before completing this notebook. Use your selected dataset and commit your outputs.

The starter code uses basic Python ideas plus a small amount of `pandas` for tables and `matplotlib` for charts. You are expected to adapt the examples, add your own indicators, and explain your choices in the report.

## 1. Load Selected Dataset

Use the manifest in `data/selected/variant_manifest.json` to load your dataset.

The manifest records which dataset variant belongs to this repository. Keeping it means your analysis can be reproduced later by your marker.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))

from iot_analysis import count_anomalies, identify_risk_windows, load_rows, numeric_summary
from variant_selector import MANIFEST_PATH, generate_dataset, select_variant

manifest = select_variant()
dataset_path = generate_dataset(manifest)
rows = load_rows(dataset_path)
df = pd.read_csv(dataset_path, parse_dates=['timestamp'])

manifest, df.head()

## 2. Explore And Summarise

Calculate summary statistics and identify important events. Add your own analysis cells as needed.

Think about which readings are sensor evidence, which readings show actuator behaviour, which readings show device or gateway quality, and which readings may need cleaning before interpretation.

The next cell gives you a first view of the data. Use it to decide what extra exploration you need.

In [ ]:
# Helper functions give a first summary without requiring complex pandas code.
summary = numeric_summary(rows)
anomaly_counts = count_anomalies(rows)
risk_windows = identify_risk_windows(rows)

# Find columns with missing values. Missing readings matter in IoT because
# a dashboard or controller may make poor decisions from incomplete data.
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]

# Count how many records belong to each greenhouse zone.
zone_counts = df['zone'].value_counts()

summary, anomaly_counts, len(risk_windows), missing_values, zone_counts

## 3. Derive IoT Indicators

Create useful operational indicators from the raw data. You may change the thresholds below if you justify the choice in your report.

Examples:

- `thermal_risk`: temperature is outside the desired growing range.
- `water_stress`: soil moisture is below the level that should trigger attention.
- `air_quality_risk`: CO2 readings may indicate poor ventilation.
- `network_degraded`: gateway latency or packet loss may affect timely monitoring/control.
- `device_health_risk`: battery, signal, or missing sensor readings may affect trust in the data.
- `edge_action_priority`: a local fog/edge action may be needed because environment and network symptoms overlap.

In [ ]:
thresholds = {
    'temperature_high_c': 27.0,
    'temperature_low_c': 18.0,
    'soil_low_pct': 42.0,
    'co2_high_ppm': 560.0,
    'latency_high_ms': 140.0,
    'packet_loss_high_pct': 2.0,
    'battery_low_pct': 82.0,
    'rssi_weak_dbm': -72.0,
}

# Each line below creates a True/False column.
# True means that row meets the condition and may need attention.
too_hot = df['air_temperature_c'] > thresholds['temperature_high_c']
too_cold = df['air_temperature_c'] < thresholds['temperature_low_c']
df['thermal_risk'] = too_hot | too_cold

# Missing soil readings are included as water-stress evidence because
# the system cannot safely confirm whether irrigation is needed.
missing_soil = df['soil_moisture_pct'].isna()
soil_too_low = df['soil_moisture_pct'] < thresholds['soil_low_pct']
df['water_stress'] = missing_soil | soil_too_low

df['air_quality_risk'] = df['co2_ppm'] > thresholds['co2_high_ppm']

high_latency = df['gateway_latency_ms'] > thresholds['latency_high_ms']
high_packet_loss = df['packet_loss_pct'] > thresholds['packet_loss_high_pct']
df['network_degraded'] = high_latency | high_packet_loss

low_battery = df['device_battery_pct'] < thresholds['battery_low_pct']
weak_signal = df['gateway_rssi_dbm'] < thresholds['rssi_weak_dbm']
sensor_not_ok = df['sensor_status'] != 'ok'
df['device_health_risk'] = low_battery | weak_signal | sensor_not_ok

# An actuator mismatch means the sensor readings suggest action, but
# the relevant actuator is not active in that row.
fan_needed = df['thermal_risk'] | df['air_quality_risk']
fan_not_running = df['fan_active'] == 0
irrigation_not_running = df['irrigation_active'] == 0
df['actuator_mismatch'] = (fan_needed & fan_not_running) | (df['water_stress'] & irrigation_not_running)

# Edge action is most important when the environment needs attention and
# the network or device health makes cloud-only control less reliable.
environment_needs_attention = df['thermal_risk'] | df['water_stress'] | df['air_quality_risk']
remote_control_less_reliable = df['network_degraded'] | df['device_health_risk']
df['edge_action_priority'] = environment_needs_attention & remote_control_less_reliable

# TODO: Add at least one additional derived indicator that is meaningful for greenhouse operations.
# For example, estimate irrigation efficiency, compare zone differences, or score recovery time after an anomaly.

df[['timestamp', 'zone', 'air_temperature_c', 'soil_moisture_pct', 'co2_ppm', 'gateway_latency_ms', 'packet_loss_pct', 'thermal_risk', 'water_stress', 'air_quality_risk', 'network_degraded', 'device_health_risk', 'actuator_mismatch', 'edge_action_priority']].head()

## 4. Visualise And Select Evidence

Create charts or tables that support your report. Save important outputs into `outputs/`. Your final repository should contain at least four useful output files that you refer to in your report, including at least one output you create or materially adapt beyond the starter examples.

Choose evidence that helps answer an IoT question, such as when local control is needed, whether actuator behaviour matches sensor conditions, or whether network quality affects monitoring.

In [ ]:
outputs_dir = ROOT / 'outputs'
outputs_dir.mkdir(exist_ok=True)

# Example output table: rows needing attention.
# This separates the filter condition from the column list so it is easier to edit.
risk_condition = df['thermal_risk'] | df['water_stress'] | df['air_quality_risk'] | df['network_degraded'] | df['device_health_risk']

risk_columns = [
    'timestamp', 'zone', 'air_temperature_c', 'relative_humidity_pct', 'soil_moisture_pct',
    'co2_ppm', 'gateway_latency_ms', 'packet_loss_pct', 'gateway_rssi_dbm',
    'device_battery_pct', 'fan_active', 'irrigation_active', 'thermal_risk',
    'water_stress', 'air_quality_risk', 'network_degraded', 'device_health_risk',
    'actuator_mismatch', 'edge_action_priority', 'anomaly_label'
]

risk_table = df.loc[risk_condition, risk_columns]
risk_table.to_csv(outputs_dir / 'greenhouse_risk_windows.csv', index=False)

# Example summary table by greenhouse zone.
zone_summary = df.groupby('zone')[['air_temperature_c', 'soil_moisture_pct', 'co2_ppm']].mean().round(2)
zone_summary['edge_action_priority_count'] = df.groupby('zone')['edge_action_priority'].sum()
zone_summary['actuator_mismatch_count'] = df.groupby('zone')['actuator_mismatch'].sum()
zone_summary.to_csv(outputs_dir / 'greenhouse_zone_summary.csv')

# Example chart 1: temperature over time by zone.
fig, ax = plt.subplots(figsize=(10, 5))
for zone, zone_df in df.groupby('zone'):
    ax.plot(zone_df['timestamp'], zone_df['air_temperature_c'], label=zone)
ax.set_title('Air temperature by greenhouse zone')
ax.set_ylabel('Air temperature (C)')
ax.set_xlabel('Timestamp')
ax.tick_params(axis='x', rotation=30)
ax.legend()
fig.tight_layout()
fig.savefig(outputs_dir / 'temperature_by_zone.png', dpi=150)
plt.close(fig)

# Example chart 2: average gateway latency over time.
latency_by_time = df.groupby('timestamp')['gateway_latency_ms'].mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(latency_by_time.index, latency_by_time.values, color='black')
ax.set_title('Average gateway latency')
ax.set_ylabel('Latency (ms)')
ax.set_xlabel('Timestamp')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(outputs_dir / 'gateway_latency_over_time.png', dpi=150)
plt.close(fig)

# TODO: Create at least one additional chart or table that you will use in your report.
# At least one output should show whether actuator behaviour is appropriate for the sensor conditions.
risk_table.head()

## 5. Interpret

Write brief notebook notes here, then develop the full explanation in your Brightspace report.

- What operational risks are visible?
- What data-management requirements does the scenario imply?
- What hardware/platform choices are suitable?
- What privacy and security risks should be managed?